# 05 — Player Value vs Performance 
This notebook fixes leakage, adds proper modelling, clustering, and interpretation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


## Load Dataset

In [ ]:
df = pd.read_csv('j1_player_value_statsbomb_merged.csv')
df.head()

## Feature Selection (No Leakage)

In [ ]:
leakage_cols = ['market_value_eur','log_market_value']
id_cols = ['player','player_name','team']

features = [c for c in df.columns if c not in leakage_cols + id_cols]

X = df[features]
y = df['log_market_value']

print(len(features), 'features used')

## Train Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train,y_train)

pred = rf.predict(X_test)

mae = mean_absolute_error(y_test,pred)
rmse = np.sqrt(mean_squared_error(y_test,pred))
r2 = r2_score(y_test,pred)

print('MAE:',mae)
print('RMSE:',rmse)
print('R2:',r2)

## Feature Importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(10).plot(kind='barh')
plt.title('Top Features')
plt.show()

## Overrated / Underrated Players

In [ ]:
df['predicted'] = rf.predict(X)
df['gap'] = df['predicted'] - df['log_market_value']

underrated = df.sort_values('gap', ascending=False).head(5)
overrated = df.sort_values('gap').head(5)

print('Underrated')
print(underrated[['player_name','gap']])

print('\nOverrated')
print(overrated[['player_name','gap']])

## Clustering (Player Archetypes)

In [ ]:
cluster_features = [c for c in df.columns if '_per90' in c]

if len(cluster_features) == 0:
    cluster_features = X.columns.tolist()

X_cluster = df[cluster_features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

df[['player_name','cluster']].head()

## PCA Visualization

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

plt.scatter(coords[:,0], coords[:,1], c=df['cluster'])
plt.title('Player Clusters')
plt.show()